In [1]:
!pip install langchain
!pip install faiss-cpu
!pip install pypdf
!pip install sentence-transformers
!pip install google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.3/347.3 kB 11.5 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import os
print(os.listdir())

In [ ]:
!pip install pypdf

In [ ]:
from pypdf import PdfReader

reader = PdfReader("real.pdf")

text = ""

for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted

print(text[:2000])  # Shows the first 2000 characters

In [ ]:
!pip install langchain

In [ ]:
!pip install -q langchain

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(text)

print("Number of chunks:", len(chunks))

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print(embeddings.shape)

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Stored", index.ntotal, "chunks")

In [ ]:
question= "what is my name?"

In [ ]:
question_embedding = model.encode([question])

In [ ]:
D, I = index.search(
    np.array(question_embedding),
    k=3
)

print(I)

In [ ]:
retrieved_chunks = [chunks[i] for i in I[0]]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Chunk {i+1} ---\n")
    print(chunk)

In [ ]:
from google.colab import files
uploaded = files.upload

In [ ]:
import os
print(os.listdir())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.listdir())

In [ ]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".pdf"):
            print(os.path.join(root, file))

In [ ]:
from pypdf import PdfReader

reader = PdfReader("/content/drive/MyDrive/Me/s41598-026-37064-2.pdf")

print(reader.pages[0].extract_text()[:1000])

In [ ]:
from pypdf import PdfReader

reader = PdfReader("/content/drive/MyDrive/Me/s41598-026-37064-2.pdf")

text = ""

for page in reader.pages:
    extracted = page.extract_text()
    if extracted:
        text += extracted

print("Characters:", len(text))

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_text(text)

print("Chunks:", len(chunks))

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print(embeddings.shape)

In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Stored", index.ntotal, "chunks")

In [ ]:
question = "What is Plus Disease in ROP?"

question_embedding = model.encode([question])

D, I = index.search(
    np.array(question_embedding),
    k=3
)

retrieved_chunks = [chunks[i] for i in I[0]]

for i, chunk in enumerate(retrieved_chunks):
    print(f"\n--- Chunk {i+1} ---\n")
    print(chunk)

In [ ]:
!pip install -q transformers sentencepiece torch

In [ ]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

In [ ]:
result = generator(
    "What is Retinopathy of Prematurity?",
    max_new_tokens=50
)

print(result[0]["generated_text"])

In [ ]:
context = "\n".join(retrieved_chunks)

prompt = f"""
Context:
{context}

Question:
What is Plus Disease in ROP?

Answer:
"""

result = generator(
    prompt,
    max_new_tokens=100
)

print(result[0]["generated_text"])

In [ ]:
question = input("Ask a question: ")

question_embedding = model.encode([question])

D, I = index.search(
    np.array(question_embedding),
    k=3
)

retrieved_chunks = [chunks[i] for i in I[0]]

context = "\n".join(retrieved_chunks)

prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

result = generator(
    prompt,
    max_new_tokens=150
)

print("\nAnswer:")
print(result[0]["generated_text"])

In [ ]:
question = input("Ask a question: ")

question_embedding = model.encode([question])

D, I = index.search(
    np.array(question_embedding),
    k=3
)

retrieved_chunks = [chunks[i] for i in I[0]]

context = "\n".join(retrieved_chunks)

result = generator(
    f"Answer based on the context:\n{context}\nQuestion: {question}",
    max_new_tokens=100
)

print(result[0]["generated_text"])

In [ ]:
%%writefile app.py

import streamlit as st

st.title("ROP RAG Chatbot")

question = st.text_input("Ask a question about ROP")

if st.button("Ask"):
    st.write("Answer will appear here")

In [ ]:
!ls

In [ ]:
!cat app.py

In [ ]:
import streamlit as st

st.title("ROP RAG Chatbot")

question = st.text_input("Ask a question about ROP")

if st.button("Ask"):
    st.write("Answer will appear here")

In [ ]:
!pip install -q streamlit

In [ ]:
question = input("Ask a question: ")

question_embedding = model.encode([question])

D, I = index.search(np.array(question_embedding), k=3)

retrieved_chunks = [chunks[i] for i in I[0]]

print("\nAnswer:\n")
print("\n".join(retrieved_chunks))

In [ ]:
question = input("Ask a question: ")

question_embedding = model.encode([question])

D, I = index.search(np.array(question_embedding), k=1)

best_chunk = chunks[I[0][0]]

print("\nAnswer:\n")
print(best_chunk)